# Put into Tasks

Take the shuffled word lists and build different versions of the html tasks.

In [3]:
def insert_words_into_html(template_path, word_list, output_path):
    """
    Insert a list of words into the sampleWords array of an HTML template.
    """

    # Read template HTML
    with open(template_path, "r", encoding="utf-8") as f:
        html = f.read()

    # Convert word list into JS array format
    words_js = json.dumps(word_list, ensure_ascii=False, indent=2)

    new_block = f"const sampleWords = {words_js};"

    # Replace the sampleWords block
    import re
    html = re.sub(
        r"const sampleWords\s*=\s*\[[\s\S]*?\];",
        new_block,
        html
    )

    # Write new HTML
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"Created {output_path}")

# Build the HTML files

This part will build the html files with the word lists and translate the words for the experiment. 

In [4]:
def generate_html_tasks(
    template_dir,
    output_dir,
    translation_csv,
    word_lists,
    lang,
    save_wordlists=True
):

    translations = pd.read_csv(translation_csv)

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    templates = sorted(Path(template_dir).glob("*.html"))

    # SAVE WORD LISTS
    if save_wordlists:
        stimuli_dir = Path(output_dir) / "stimuli"
        stimuli_dir.mkdir(exist_ok=True)

        for i, words in enumerate(word_lists, start=1):
            pd.DataFrame({"word": words}).to_csv(
                stimuli_dir / f"wordlist_{lang}_{i}.csv",
                index=False
            )

    for template in templates:

        with open(template, "r", encoding="utf-8") as f:
            html = f.read()

        # Replace UI strings
        for _, row in translations.iterrows():
            english = str(row["english"])
            translated = str(row[f"{lang}"])
            html = html.replace(english, translated)

        # Generate HTML with word lists
        for i, word_list in enumerate(word_lists, start=1):

            words_js = json.dumps(word_list, ensure_ascii=False, indent=2)

            new_block = f"const sampleWords = {words_js};"

            html_with_words = re.sub(
                r"const sampleWords\s*=\s*\[[\s\S]*?\];",
                new_block,
                html
            )

            outfile = (
                Path(output_dir)
                / f"{template.stem}_{lang}_{i}.html"
            )

            with open(outfile, "w", encoding="utf-8") as f:
                f.write(html_with_words)

            # Save translation transparency file
            translation_snapshot = translations[["id", "english", lang]]
            
            snapshot_path = Path(output_dir) / f"translation_map_{lang}.csv"
            
            translation_snapshot.to_csv(snapshot_path, index=False)

            print("Created", outfile)


# Build .jas Files

These files are required to make a valid jzip for JATOS.

In [5]:
def generate_jas(template_path, output_path, experiment, lang, version):

    with open(template_path, "r") as f:
        jas = json.load(f)

    name = f"{experiment}_{lang}_{version}"

    # new UUIDs
    study_uuid = str(uuid.uuid4())
    component_uuid = str(uuid.uuid4())
    batch_uuid = str(uuid.uuid4())

    jas["data"]["uuid"] = study_uuid
    jas["data"]["title"] = name
    jas["data"]["description"] = f"{experiment} study ({lang}) version {version}"
    jas["data"]["dirName"] = name

    jas["data"]["componentList"][0]["uuid"] = component_uuid
    jas["data"]["componentList"][0]["title"] = name

    jas["data"]["batchList"][0]["uuid"] = batch_uuid

    with open(output_path, "w") as f:
        json.dump(jas, f, indent=2)

    print("Created", output_path)

# Libraries and Models

In [30]:
import pandas as pd
from transformers import AutoProcessor, AutoModelForSeq2SeqLM
import random
from pathlib import Path
import json
import uuid
from pathlib import Path
from nltk.corpus import wordnet
import re

# Load model once (important for speed)
model_name = "facebook/seamless-m4t-v2-large"

processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

word_pairs_english = pd.read_csv("en_words.csv")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Afrikaans

In [33]:
# first translate the words
# translate_pairs(word_pairs_english, "afr")

# second manually edit files save as _update
word_lists = build_word_lists("afr/afr_word_pairs_update.csv", "afr")


2000 total stimuli
1915 unique stimuli
5 word lists created for afr
